# 09. 외부 프레임워크 비교

## 학습 목표
- Deep Agents, LangGraph, LangChain의 심화 차이를 이해한다
- OpenCode, Claude Agent SDK와 비교한다
- 아키텍처, 유연성, 생태계를 비교 분석한다
- 사용 사례별 추천 프레임워크를 안다
- 마이그레이션 고려사항을 이해한다

In [ ]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"모델 설정 완료: {model.model_name}")

---
## 1. 비교 개요

AI 에이전트 프레임워크를 선택할 때는 **모델 지원**, **아키텍처**, **생태계**, **라이선스** 등 여러 요소를 고려해야 합니다.

이 노트북에서는 세 가지 주요 프레임워크를 비교합니다:

- **LangChain Deep Agents** — 모델 무관(model-agnostic) 에이전트 하네스
- **OpenCode** — 터미널/데스크톱/IDE 기반 코딩 에이전트
- **Claude Agent SDK** — Anthropic의 Claude 전용 에이전트 SDK

---
## 2. Deep Agents vs OpenCode vs Claude Agent SDK

### 기본 비교

| 특성 | LangChain Deep Agents | OpenCode | Claude Agent SDK |
|------|----------------------|----------|------------------|
| **모델 지원** | 모델 무관 (Anthropic, OpenAI, 100+ 제공자) | 75+ 제공자 (Ollama 포함 로컬) | Claude 모델 전용 |
| **라이선스** | MIT | MIT | MIT (SDK), 독점 (Claude Code) |
| **SDK** | Python, TypeScript + CLI | 터미널, 데스크톱, IDE 확장 | Python, TypeScript |
| **샌드박스** | 통합 도구로 사용 가능 | 미지원 | 미지원 |
| **상태 관리** | 타임 트래블 지원 | 미지원 | 타임 트래블 지원 |
| **Observability** | LangSmith 네이티브 | 없음 | 없음 |

In [ ]:
# 프레임워크 비교 테이블 출력
frameworks = {
    "LangChain Deep Agents": {
        "모델 지원": "100+ 제공자 (model-agnostic)",
        "라이선스": "MIT",
        "SDK": "Python, TypeScript, CLI",
        "샌드박스": "통합 지원",
        "타임 트래블": "지원",
    },
    "OpenCode": {
        "모델 지원": "75+ 제공자 (로컬 포함)",
        "라이선스": "MIT",
        "SDK": "터미널, 데스크톱, IDE",
        "샌드박스": "미지원",
        "타임 트래블": "미지원",
    },
    "Claude Agent SDK": {
        "모델 지원": "Claude 전용",
        "라이선스": "MIT (SDK)",
        "SDK": "Python, TypeScript",
        "샌드박스": "미지원",
        "타임 트래블": "지원",
    },
}

print("=== 프레임워크 비교 ===")
for name, features in frameworks.items():
    print(f"\n[{name}]")
    for key, value in features.items():
        print(f"  {key}: {value}")

---
## 3. 핵심 기능 비교

### 공통 기능
세 프레임워크 모두 다음 기능을 지원합니다:
- 파일 작업 (읽기, 쓰기, 편집)
- 쉘 명령 실행
- 검색 기능 (grep, glob)
- 계획 기능 (태스크 리스트)
- Human-in-the-Loop (권한 프레임워크는 상이)

### 차별화 기능

| 기능 | Deep Agents | OpenCode | Claude Agent SDK |
|------|------------|----------|------------------|
| **코어 도구** | 파일, 쉘, 검색, 계획 | 파일, 쉘, 검색, 계획 | 파일, 쉘, 검색, 계획 |
| **샌드박스 통합** | 도구로 통합 가능 | 없음 | 없음 |
| **플러거블 백엔드** | 스토리지, 파일시스템 | 없음 | 없음 |
| **가상 파일시스템** | 플러거블 백엔드 | 없음 | 없음 |
| **네이티브 트레이싱** | LangSmith | 없음 | 없음 |

---
## 4. 아키텍처 비교

### LangChain Deep Agents
- **플러거블 스토리지 백엔드** — 상태, 파일시스템, 스토어를 독립적으로 구성
- **가상 파일시스템** — 로컬, 인메모리, 샌드박스 백엔드 교체 가능
- **LangGraph 기반** — 그래프 실행 엔진으로 복잡한 워크플로 지원
- **미들웨어 시스템** — 에이전트 동작을 세밀하게 커스터마이징

### OpenCode
- **터미널 네이티브** — 가볍고 빠른 시작
- **75+ 모델 제공자** — Ollama를 통한 로컬 모델 지원
- **LSP 통합** — 코드 편집에 특화된 에디터 기능

### Claude Agent SDK
- **Claude 최적화** — Claude 모델에 특화된 기능
- **타임 트래블** — 상태 분기(branching) 지원
- **간결한 API** — 빠른 프로토타이핑에 적합

---
## 5. 사용 사례별 추천

| 사용 사례 | 추천 프레임워크 | 이유 |
|----------|--------------|------|
| 프로덕션 에이전트 앱 | **Deep Agents** | 플러거블 백엔드, 옵저버빌리티, 샌드박스 |
| 멀티 모델 에이전트 | **Deep Agents** | 100+ 모델 제공자 지원 |
| 터미널 코딩 어시스턴트 | **OpenCode** | 가볍고 빠른 시작, 로컬 모델 |
| Claude 전용 앱 | **Claude Agent SDK** | Claude 최적화, 간결한 API |
| 빠른 프로토타이핑 | **Claude Agent SDK** | 간결한 API, 빠른 설정 |
| 복잡한 멀티 에이전트 시스템 | **Deep Agents** | 서브에이전트, 컨텍스트 관리 |
| 로컬 모델 사용 | **OpenCode** | Ollama 네이티브 지원 |

In [ ]:
# 사용 사례별 프레임워크 추천 도우미
def recommend_framework(use_case: str) -> str:
    """사용 사례에 따라 프레임워크를 추천합니다."""
    recommendations = {
        "production": ("Deep Agents", "플러거블 백엔드, 옵저버빌리티, 샌드박스"),
        "multi-model": ("Deep Agents", "100+ 모델 제공자 지원"),
        "terminal": ("OpenCode", "가볍고 빠른 시작, 로컬 모델"),
        "claude-only": ("Claude Agent SDK", "Claude 최적화, 간결한 API"),
        "prototyping": ("Claude Agent SDK", "간결한 API, 빠른 설정"),
        "multi-agent": ("Deep Agents", "서브에이전트, 컨텍스트 관리"),
        "local-model": ("OpenCode", "Ollama 네이티브 지원"),
    }
    if use_case in recommendations:
        fw, reason = recommendations[use_case]
        return f"{fw} — {reason}"
    return "해당 사용 사례를 찾을 수 없습니다."


# 테스트
test_cases = ["production", "terminal", "claude-only", "multi-agent"]
print("=== 프레임워크 추천 ===")
for case in test_cases:
    print(f"  {case}: {recommend_framework(case)}")

---
## 6. 생태계 비교

| 항목 | Deep Agents | OpenCode | Claude Agent SDK |
|------|------------|----------|------------------|
| **커뮤니티** | LangChain 생태계 (대규모) | GitHub 커뮤니티 | Anthropic 커뮤니티 |
| **문서** | 공식 문서 + LangSmith 연동 | GitHub README | Anthropic 공식 문서 |
| **통합** | LangChain, LangGraph, LangSmith | LSP, 터미널 | Claude API |
| **패키지 관리** | pip/uv | go install / brew | pip/npm |
| **에디터 통합** | ACP (Zed, JetBrains, VS Code, Neovim) | 자체 에디터 | 없음 |

---
## 7. 마이그레이션 고려사항

프레임워크 간 마이그레이션 시 고려할 핵심 사항:

### 공통 고려사항
1. **모델 호환성** — 사용 중인 모델이 대상 프레임워크에서 지원되는지 확인
2. **도구 호환성** — 커스텀 도구의 인터페이스 변환 필요
3. **상태 관리** — 체크포인트/메모리 마이그레이션 방법 확인
4. **옵저버빌리티** — 트레이싱/로깅 솔루션 대체 방안

### Deep Agents로 마이그레이션 시 장점
- **LangChain 도구 재사용** — 기존 LangChain 도구를 그대로 사용 가능
- **LangGraph 호환** — LangGraph 그래프와 연동 가능
- **점진적 마이그레이션** — 기존 코드를 단계적으로 전환 가능

### 주의사항
- Claude Agent SDK에서 마이그레이션 시 Claude 전용 기능은 대체 구현 필요
- OpenCode에서 마이그레이션 시 터미널 UI 로직 분리 필요

---
## 전체 요약

| 주제 | 핵심 개념 | 핵심 API/도구 |
|------|----------|-------------|
| 3-way 비교 | Deep Agents, OpenCode, Claude Agent SDK | 모델 지원, 라이선스, SDK |
| 핵심 기능 | 공통 도구 + 차별화 기능 | 샌드박스, 플러거블 백엔드 |
| 아키텍처 | 플러거블 vs 네이티브 vs 최적화 | LangGraph, LSP, Claude API |
| 사용 사례 | 프로덕션, 터미널, 프로토타이핑 등 | `recommend_framework()` |
| 생태계 | 커뮤니티, 문서, 통합, 에디터 | LangSmith, ACP |
| 마이그레이션 | 모델/도구/상태 호환성 확인 | 점진적 전환 |

### 다음 단계
→ **[10. 샌드박스와 ACP](10_sandboxes_and_acp.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [Comparison with OpenCode and Claude Agent SDK](../docs/deepagents/04-comparison.md)